# End of week Exercise - week 2

Built a Gradio-based AI Technical Tutor that lets users select an LLM (GPT, Claude, or Gemini) and an audience profile to dynamically tailor responses. It maintains conversational context with streaming responses and automatically resets the conversation when the selected model or audience changes.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")


In [ ]:
openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)


MODELS = {
    "GPT": {
        "client": openai,
        "model": "gpt-4.1-mini"
    },
    "Claude": {
        "client": anthropic,
        "model": "claude-haiku-4-5"
    },
    "Gemini": {
        "client": gemini,
        "model": "gemini-3.1-flash-lite"
    }
}

In [ ]:
# audience: "beginner" | "intermediate" | "java-developer" | "interview"
AUDIENCE_INSTRUCTIONS = {
    "beginner": (
        "Explain like to a smart beginner. Use simple language, short analogies, "
        "and avoid jargon unless you define it. Show a tiny concrete example."
    ),

    "intermediate": (
        "Assume solid Python basics. Be precise about semantics, edge cases, and "
        "when you'd use this pattern in real code. Keep it concise."
    ),

    "java-developer": (
        "Assume solid Python and Java basics. Use Java terms and examples to explain "
        "if possible. Be precise about semantics, edge cases, and when you'd use "
        "this pattern in real code. Keep it concise. Show a tiny concrete example "
        "in both languages."
    ),

    "interview": (
        "Answer as for a coding interview. Cover what it does, time/space notes "
        "if relevant, common pitfalls, and a crisp example. Prefer clarity over fluff."
    ),

    "example-only": (
        "Just show an example of code in java, do not use additional text. If you can't do it, just say so and answer in no more than 3 sentences why."
    ),
}

def build_system_prompt(audience):

    return f"""
You are a helpful IT technical tutor for Python, Java, software engineering,
data science, and LLMs.

Respond in markdown.

Structure your answer as:

1. What it does
2. Why / when you'd use it
3. Step-by-step
4. Pitfalls
5. Tiny example

Audience guidance:

{AUDIENCE_INSTRUCTIONS[audience]}
"""

In [ ]:
def ask_llm(question, model_name, audience, history):

    client = MODELS[model_name]["client"]
    model = MODELS[model_name]["model"]

    system_prompt = build_system_prompt(audience)

    messages = [
        {
            "role": "system",
            "content": system_prompt
        }
    ]
    messages.extend(history)
    messages.append({
        "role": "user",
        "content": question
    })

    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )

    for chunk in stream:

        if not chunk.choices:
            continue

        content = chunk.choices[0].delta.content

        if content:
            yield content

In [ ]:
def clear_history():
    return [], []

In [ ]:
def chat(question, model, audience, history):

    if not question.strip():
        yield history, ""
        return

    previous_history = history.copy()

    history.append({
        "role": "user",
        "content": question
    })

    history.append({
        "role": "assistant",
        "content": ""
    })

    yield history, ""

    answer = ""

    # Stream response from the LLM.
    for chunk in ask_llm(
        question=question,
        model_name=model,
        audience=audience,
        history=previous_history
    ):

        answer += chunk

        history[-1]["content"] = answer

        yield history, ""

In [ ]:
with gr.Blocks(title="AI Technical Tutor") as demo:

    gr.Markdown("# 🤖 AI Technical Tutor")

    with gr.Row():

        model_dropdown = gr.Dropdown(
            choices=["GPT", "Claude", "Gemini"],
            value="GPT",
            label="Model"
        )

        audience_dropdown = gr.Dropdown(
            choices=list(AUDIENCE_INSTRUCTIONS.keys()),
            value="java-developer",
            label="Audience"
        )

    chatbot = gr.Chatbot(
        label="Conversation",
        type="messages"
    )

    question = gr.Textbox(
        label="Your question",
        placeholder="Ask a technical question...",
        lines=3
    )

    submit = gr.Button(
        "Ask",
        variant="primary"
    )

    history = gr.State([])

    # Ask
    submit.click(
        fn=chat,
        inputs=[
            question,
            model_dropdown,
            audience_dropdown,
            history
        ],
        outputs=[
            chatbot,
            question
        ]
    )

    # Changing model clears conversation
    model_dropdown.change(
        fn=clear_history,
        inputs=[],
        outputs=[
            chatbot,
            question
        ]
    ).then(
        fn=lambda: [],
        inputs=[],
        outputs=history
    )

    # Changing audience clears conversation
    audience_dropdown.change(
        fn=clear_history,
        inputs=[],
        outputs=[
            chatbot,
            question
        ]
    ).then(
        fn=lambda: [],
        inputs=[],
        outputs=history
    )


demo.launch()